In [14]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os

# Tắt cảnh báo chia cho 0 do dữ liệu mất cân bằng
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# ==========================================
# 1. ĐỌC DỮ LIỆU & CHUẨN BỊ LỚP
# ==========================================
df = pd.read_csv('/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv')

print(" Phân bố điểm chất lượng gốc (quality):")
print(df['quality'].value_counts().sort_index())

def map_quality_group(q):
    if q <= 4: return 0
    elif q <= 6: return 1
    else: return 2

df['quality_group'] = df['quality'].apply(map_quality_group)
print("\n Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):")
print(df['quality_group'].value_counts().sort_index())

# ==========================================
# 2. CHIA DỮ LIỆU & CHUẨN HÓA
# ==========================================
X = df.drop(columns=['Id', 'quality', 'quality_group'])
y = df['quality_group']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\n Đã chuẩn hóa dữ liệu thành công.")

os.makedirs('/kaggle/working', exist_ok=True)

# ==========================================
# 3. HUẤN LUYỆN & ĐÁNH GIÁ RIÊNG TỪNG MÔ HÌNH
# ==========================================

# ------------------------------------------
# 3.1 LOGISTIC REGRESSION
# ------------------------------------------
print("\n" + "="*50)
print(" MÔ HÌNH 1: LOGISTIC REGRESSION")
print("="*50)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)

print(f" Accuracy : {acc_lr:.4f}")
print(f" F1-Score : {f1_lr:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(lr, '/kaggle/working/wine_quality_logreg.pkl')
print(" Đã lưu: /kaggle/working/wine_quality_logreg.pkl")

# ------------------------------------------
# 3.2 RANDOM FOREST
# ------------------------------------------
print("\n" + "="*50)
print(" MÔ HÌNH 2: RANDOM FOREST")
print("="*50)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

print(f" Accuracy : {acc_rf:.4f}")
print(f" F1-Score : {f1_rf:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(rf, '/kaggle/working/wine_quality_rf.pkl')
print("💾 Đã lưu: /kaggle/working/wine_quality_rf.pkl")

# ------------------------------------------
# 3.3 KNN 
# ------------------------------------------
print("\n" + "="*50)
print(" BƯỚC TỐI ƯU: TÌM K TỐT NHẤT CHO KNN (CV trên tập TRAIN)")
print("="*50)

from sklearn.model_selection import cross_val_score

best_k = 5
best_cv_score = 0

for k in range(3, 15):
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # ✅ SỬA: Dùng cross-validation trên tập TRAIN, KHÔNG dùng test set
    cv_scores = cross_val_score(knn_temp, X_train_scaled, y_train, cv=5, scoring='f1_weighted')
    mean_cv_score = cv_scores.mean()
    print(f"  k={k} | CV F1-Score: {mean_cv_score:.4f} (+/- {cv_scores.std():.4f})")
    
    if mean_cv_score > best_cv_score:
        best_cv_score = mean_cv_score
        best_k = k

print(f"\n✅ Chọn k = {best_k} cho KNN (CV F1-Score: {best_cv_score:.4f})")

print("\n" + "="*50)
print(" MÔ HÌNH 3: KNN")
print("="*50)

knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted', zero_division=0)

print(f" Accuracy : {acc_knn:.4f}")
print(f" F1-Score : {f1_knn:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_knn, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(knn, '/kaggle/working/wine_quality_knn.pkl')
print(" Đã lưu: /kaggle/working/wine_quality_knn.pkl")

# ==========================================
#  4. LỰA CHỌN & LƯU MÔ HÌNH TỐT NHẤT
# ==========================================
print("\n" + "="*50)
print(" TỔNG HỢP & CHỌN MODEL TỐT NHẤT")
print("="*50)

# Gom kết quả vào dictionary để dễ so sánh
models_results = {
    "Logistic Regression": {"model": lr, "acc": acc_lr, "f1": f1_lr},
    "Random Forest": {"model": rf, "acc": acc_rf, "f1": f1_rf},
    f"KNN (k={best_k})": {"model": knn, "acc": acc_knn, "f1": f1_knn}
}

# Chọn model có F1-Score cao nhất (ưu tiên F1 vì dữ liệu mất cân bằng)
best_model_name = max(models_results, key=lambda x: models_results[x]["f1"])
best_model = models_results[best_model_name]["model"]

# Lưu model tốt nhất
joblib.dump(best_model, '/kaggle/working/wine_quality_best_model.pkl')

print(" Đã lưu file thứ 4: /kaggle/working/wine_quality_best_model.pkl")

# ==========================================
# 5. LƯU SCALER & TỔNG KẾT
# ==========================================
joblib.dump(scaler, '/kaggle/working/scaler.pkl')
print("\n Đã lưu scaler: /kaggle/working/scaler.pkl")

print("\n" + "="*65)
print(" BẢNG TỔNG KẾT KẾT QUẢ (4 FILES ĐÃ LƯU)")
print("="*65)
print(f"{'Mô hình':<25} | {'Accuracy':<10} | {'F1-Score':<10} | {'Đánh giá'}")
print("-" * 70)
for name, res in models_results.items():
    badge = " TỐT NHẤT" if name == best_model_name else ""
    print(f"{name:<25} | {res['acc']:<10.4f} | {res['f1']:<10.4f} | {badge}")



 Phân bố điểm chất lượng gốc (quality):
quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

 Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):
quality_group
0     39
1    945
2    159
Name: count, dtype: int64

 Đã chuẩn hóa dữ liệu thành công.

 MÔ HÌNH 1: LOGISTIC REGRESSION
 Accuracy : 0.5895
 F1-Score : 0.6581

 Classification Report:
              precision    recall  f1-score   support

         Kém       0.08      0.62      0.15         8
  Trung bình       0.94      0.55      0.69       189
         Tốt       0.45      0.81      0.58        32

    accuracy                           0.59       229
   macro avg       0.49      0.66      0.47       229
weighted avg       0.84      0.59      0.66       229

 Đã lưu: /kaggle/working/wine_quality_logreg.pkl

 MÔ HÌNH 2: RANDOM FOREST
 Accuracy : 0.8996
 F1-Score : 0.8810

 Classification Report:
              precision    recall  f1-score   support

         Kém       0.00      0.00     

In [16]:
from joblib import load

model = load('/kaggle/working/wine_quality_best_model.pkl')
scaler = load('/kaggle/working/scaler.pkl')

In [19]:
import pandas as pd
import numpy as np

# Tạo dữ liệu mẫu để test
new_wine = pd.DataFrame({
    'fixed acidity': [3.4, 9.8],
    'volatile acidity': [0.74, 0.86],
    'citric acid': [0.01, 0.00],
    'residual sugar': [1.7, 1.6],
    'chlorides': [0.06, 0.08],
    'free sulfur dioxide': [9.0, 15.0],
    'total sulfur dioxide': [24.0, 6.0],
    'density': [0.9978, 0.9968],
    'pH': [3.51, 3.20],
    'sulphates': [0.56, 0.68],
    'alcohol': [9.4, 9.8]
})

# Scale dữ liệu
new_wine_scaled = scaler.transform(new_wine)

# Dự đoán
predictions = model.predict(new_wine_scaled)
probabilities = model.predict_proba(new_wine_scaled)

# Hiển thị kết quả
for i in range(len(new_wine)):
    print(f"\nMẫu {i+1}:")
    print(f"  Dự đoán chất lượng: {predictions[i]}")
    print(f"  Độ tin cậy: {max(probabilities[i])*100:.2f}%")


Mẫu 1:
  Dự đoán chất lượng: 1
  Độ tin cậy: 78.00%

Mẫu 2:
  Dự đoán chất lượng: 1
  Độ tin cậy: 85.00%


In [22]:
import pandas as pd
import numpy as np
import pickle
import os
import joblib  # Thêm joblib để dự phòng nếu pickle bị lỗi

# 1. Cấu hình đường dẫn
WORKING_DIR = '/kaggle/working'
SCALER_PATH = os.path.join(WORKING_DIR, 'scaler.pkl')

# 2. Load Scaler
try:
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("Đã load scaler thành công!")
except Exception as e:
    print(f"Lỗi load scaler: {e}")

# 3. Tạo dữ liệu mẫu để test (2 mẫu như code của bạn)
new_wine = pd.DataFrame({
    'fixed acidity': [3.4, 9.8],
    'volatile acidity': [0.74, 0.86],
    'citric acid': [0.01, 0.00],
    'residual sugar': [1.7, 1.6],
    'chlorides': [0.06, 0.08],
    'free sulfur dioxide': [9.0, 15.0],
    'total sulfur dioxide': [24.0, 6.0],
    'density': [0.9978, 0.9968],
    'pH': [3.51, 3.20],
    'sulphates': [0.56, 0.68],
    'alcohol': [9.4, 9.8]
})

# 4. Scale dữ liệu (chỉ cần làm 1 lần)
new_wine_scaled = scaler.transform(new_wine)

# 5. Danh sách các model cần test
# Lấy danh sách file .pkl trong thư mục, loại trừ scaler.pkl
model_files = [f for f in os.listdir(WORKING_DIR) if f.endswith('.pkl') and f != 'scaler.pkl']

# 6. Vòng lặp chạy từng model
for model_file in sorted(model_files):
    model_path = os.path.join(WORKING_DIR, model_file)
    # Lấy tên model từ tên file (ví dụ: wine_quality_knn.pkl -> knn)
    model_name = model_file.replace('wine_quality_knn', '').replace('.pkl', '')
    
    print(f"\n{'='*50}")
    print(f"ĐANG CHẠY MODEL: {model_name.upper()}")
    print(f"{'='*50}")
    
    try:
        # Thử load model
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
            
        # Dự đoán
        predictions = model.predict(new_wine_scaled)
        
        # Hiển thị kết quả cho từng mẫu
        for i in range(len(new_wine)):
            print(f"\nMẫu {i+1}:")
            print(f"  Dự đoán chất lượng: {predictions[i]}")
            
            # Kiểm tra xem model có hỗ trợ tính xác suất không
            if hasattr(model, 'predict_proba'):
                probabilities = model.predict_proba(new_wine_scaled)
                print(f"  Độ tin cậy: {max(probabilities[i])*100:.2f}%")
            else:
                print("  (Model này không hỗ trợ tính độ tin cậy)")
                
    except Exception as e:
        # Nếu lỗi pickle, thử dùng joblib (thường dùng cho sklearn)
        print(f"Lỗi với pickle, thử dùng joblib...")
        try:
            model = joblib.load(model_path)
            
            predictions = model.predict(new_wine_scaled)
            
            for i in range(len(new_wine)):
                print(f"\nMẫu {i+1}:")
                print(f"  Dự đoán chất lượng: {predictions[i]}")
                
                if hasattr(model, 'predict_proba'):
                    probabilities = model.predict_proba(new_wine_scaled)
                    print(f"  Độ tin cậy: {max(probabilities[i])*100:.2f}%")
                    
        except Exception as e2:
            print(f"Model {model_name} bị lỗi hoặc file hỏng: {e2}")

Lỗi load scaler: STACK_GLOBAL requires str

ĐANG CHẠY MODEL: WINE_QUALITY_BEST_MODEL
Lỗi với pickle, thử dùng joblib...

Mẫu 1:
  Dự đoán chất lượng: 1
  Độ tin cậy: 78.00%

Mẫu 2:
  Dự đoán chất lượng: 1
  Độ tin cậy: 85.00%

ĐANG CHẠY MODEL: 
Lỗi với pickle, thử dùng joblib...

Mẫu 1:
  Dự đoán chất lượng: 1
  Độ tin cậy: 89.63%

Mẫu 2:
  Dự đoán chất lượng: 1
  Độ tin cậy: 90.53%

ĐANG CHẠY MODEL: WINE_QUALITY_LOGREG
Lỗi với pickle, thử dùng joblib...

Mẫu 1:
  Dự đoán chất lượng: 1
  Độ tin cậy: 81.94%

Mẫu 2:
  Dự đoán chất lượng: 0
  Độ tin cậy: 81.70%

ĐANG CHẠY MODEL: WINE_QUALITY_RF
Lỗi với pickle, thử dùng joblib...

Mẫu 1:
  Dự đoán chất lượng: 1
  Độ tin cậy: 78.00%

Mẫu 2:
  Dự đoán chất lượng: 1
  Độ tin cậy: 85.00%
